Identical split to training: seed 42, 90/10

In [ ]:
import json, random, math, os, gc
import numpy as np, pandas as pd, torch
 
DATASET_PATH = "tanglish.jsonl"
ADAPTER_DIR  = "tanglish_gemma2b_adapter"
BASE_ID      = "google/gemma-2b"
IT_ID        = "google/gemma-2b-it"
MAX_NEW      = 200
RESULTS_DIR  = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
 
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw = [json.loads(l) for l in f]
 
def fmt(r):
    ins, inp, out = (r.get("instruction", "").strip(), r.get("input", "").strip(), r.get("output", "").strip())
    user = f"{ins}\n\nContext:\n{inp}" if inp else ins
    return {
        "text": f"<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n{out}<end_of_turn>",
        "instruction": ins, "input": inp, "output": out,
    }
 
formatted = [fmt(r) for r in raw]
random.seed(42)
random.shuffle(formatted)
cut = int(len(formatted) * 0.9)
train, val = formatted[:cut], formatted[cut:]
print(f"Train {len(train)} | Val {len(val)}")
 
MODEL_TAG = "<start_of_turn>model\n"
def split_prompt_target(text):
    prompt = text.split(MODEL_TAG)[0] + MODEL_TAG
    target = text.split(MODEL_TAG)[-1].replace("<end_of_turn>", "").strip()
    return prompt, target
 
val_prompts, val_refs = zip(*[split_prompt_target(v["text"]) for v in val])
val_prompts, val_refs = list(val_prompts), list(val_refs)


Fuzzy leakage check (near-duplicates, not just exact)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
 
def near_dup_report(train_texts, val_texts, name, thr=0.90):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(train_texts + val_texts)
    sims = cosine_similarity(vec.transform(val_texts), vec.transform(train_texts))
    best = sims.max(axis=1)
    print(f"[{name}] mean best-sim {best.mean():.3f} | max {best.max():.3f} | "
          f"val items with sim >= {thr}: {(best >= thr).sum()}/{len(best)}")
    return best
 
best_prompt = near_dup_report([t["instruction"] + " " + t["input"] for t in train],
                              [v["instruction"] + " " + v["input"] for v in val], "prompts")
best_output = near_dup_report([t["output"] for t in train],
                              [v["output"] for v in val], "outputs")
pd.DataFrame({"prompt_sim": best_prompt, "output_sim": best_output}).to_csv(
    f"{RESULTS_DIR}/leakage_fuzzy.csv", index=False)


Load models: base+LoRA (one object, adapter toggled) and Gemma-2B-it

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
 
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE_ID)
tok.pad_token = tok.eos_token
END_ID = tok.convert_tokens_to_ids("<end_of_turn>")
 
base = AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=bnb, device_map="auto")
ft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
ft_model.eval()
 
it_model = AutoModelForCausalLM.from_pretrained(IT_ID, quantization_config=bnb, device_map="auto")
it_model.eval()
 


Core functions: response-only NLL (token-weighted) and generation

In [ ]:
import torch.nn.functional as F
 
@torch.no_grad()
def response_nll(model, text, prompt):
    """Return (sum NLL over response tokens, number of response tokens)."""
    ids = tok(text, return_tensors="pt").input_ids.to("cuda")
    p_len = len(tok(prompt).input_ids)
    logits = model(input_ids=ids).logits[:, :-1, :].float()
    targets = ids[:, 1:]
    nll = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), reduction="none")
    nll = nll.reshape(targets.shape)[:, p_len - 1:]        # response tokens only
    return nll.sum().item(), nll.numel()
 
@torch.no_grad()
def generate(model, prompt):
    inp = tok(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inp, max_new_tokens=MAX_NEW, do_sample=False,
                         eos_token_id=[tok.eos_token_id, END_ID], pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
 
def run_model(name, model, use_adapter=None):
    """use_adapter: None (plain model), True (adapter on), False (adapter off)."""
    nll_sum, ntok, preds = [], [], []
    for v, prompt in zip(val, val_prompts):
        if use_adapter is False:
            with model.disable_adapter():
                s, n = response_nll(model, v["text"], prompt)
                p = generate(model, prompt)
        else:
            s, n = response_nll(model, v["text"], prompt)
            p = generate(model, prompt)
        nll_sum.append(s); ntok.append(n); preds.append(p)
    print(f"{name}: done")
    return {"nll_sum": np.array(nll_sum), "ntok": np.array(ntok), "preds": preds}

Run all three systems on the validation set

In [ ]:
runs = {
    "base":        run_model("base", ft_model, use_adapter=False),
    "finetuned":   run_model("finetuned", ft_model, use_adapter=True),
    "gemma2b_it":  run_model("gemma2b_it", it_model, use_adapter=None),
}

Per-sample metrics: ROUGE, chrF, BERTScore

In [ ]:
import evaluate
from sacrebleu.metrics import CHRF
 
rouge = evaluate.load("rouge")
chrf = CHRF()
 
def per_sample_metrics(preds, refs):
    r = rouge.compute(predictions=preds, references=refs, use_aggregator=False)
    out = {"rouge1": np.array(r["rouge1"]), "rouge2": np.array(r["rouge2"]), "rougeL": np.array(r["rougeL"]),
           "chrf": np.array([chrf.sentence_score(p, [t]).score for p, t in zip(preds, refs)])}
    try:
        from bert_score import score as bscore
        _, _, f1 = bscore(preds, refs, model_type="bert-base-multilingual-cased", verbose=False)
        out["bertscore_f1"] = f1.numpy()
    except Exception as e:
        print("BERTScore skipped:", e)
    return out
 
for k, r in runs.items():
    r["metrics"] = per_sample_metrics(r["preds"], val_refs)
